# Action Label Dataset Inspector

This notebook inspects recorded transition data from `sample_retro_transitions.py` and focuses on **ground-truth action labels** per game run.

In [1]:
from pathlib import Path
import json
from collections import Counter, defaultdict
from retro_transition_dataloader import TransitionLoaderConfig, create_transition_dataloaders



In [2]:

DATASET_DIR = Path("data/retro_platformer_stage1_v1")
assert DATASET_DIR.exists(), f"Missing dataset dir: {DATASET_DIR}"

DATA_PATH = DATASET_DIR / 'transitions.jsonl'
assert DATA_PATH.exists(), f'Missing file: {DATA_PATH}'

records = []
with DATA_PATH.open('r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f'Loaded {len(records)} transitions from {DATA_PATH}')
print('Sample keys:', sorted(records[0].keys()))

Loaded 18000 transitions from data/retro_platformer_stage1_v1/transitions.jsonl
Sample keys: ['action_label', 'action_policy', 'action_vector', 'episode', 'frame_t', 'frame_tp1', 'game', 'platformer_reason', 'pressed_buttons', 'reward', 'step', 'system', 'terminated', 'truncated']


In [3]:
# Per-game counts and label diversity
per_game_count = Counter(r['game'] for r in records)
per_game_unique_labels = defaultdict(set)
for r in records:
    per_game_unique_labels[r['game']].add(r['action_label'])

print('Per-game transition counts and unique action labels:')
for game in sorted(per_game_count):
    print(f'- {game}: transitions={per_game_count[game]}, unique_labels={len(per_game_unique_labels[game])}')

Per-game transition counts and unique action labels:
- AdventureIslandII-Nes-v0: transitions=3000, unique_labels=10
- Castlevania-Nes-v0: transitions=3000, unique_labels=10
- NinjaGaiden-Nes-v0: transitions=3000, unique_labels=9
- SonicTheHedgehog-Sms-v0: transitions=3000, unique_labels=10
- SuperMarioBros-Nes-v0: transitions=3000, unique_labels=10
- SuperMarioBros3-Nes-v0: transitions=3000, unique_labels=10


In [4]:
# Top action labels for each game
top_k = 15
label_counts_by_game = defaultdict(Counter)
for r in records:
    label_counts_by_game[r['game']][r['action_label']] += 1

for game in sorted(label_counts_by_game):
    print(f'\n=== {game} (top {top_k} labels) ===')
    for label, count in label_counts_by_game[game].most_common(top_k):
        print(f'{count:4d}  {label}')


=== AdventureIslandII-Nes-v0 (top 15 labels) ===
1039  RIGHT
 771  RIGHT+B
 612  RIGHT+A
 181  RIGHT+A+B
 120  NOOP
  84  B
  83  A
  72  LEFT
  24  RECOVERY_RIGHT+A+B
  14  LEFT+A

=== Castlevania-Nes-v0 (top 15 labels) ===
 818  RIGHT
 693  RIGHT+B
 550  RIGHT+A
 281  RIGHT+A+B
 184  NOOP
 140  B
 122  A
 108  LEFT+A
  96  LEFT
   8  RECOVERY_RIGHT+A+B

=== NinjaGaiden-Nes-v0 (top 15 labels) ===
 904  RIGHT
 791  RIGHT+B
 580  RIGHT+A
 198  RIGHT+A+B
 143  NOOP
 125  B
 106  LEFT
  91  LEFT+A
  62  A

=== SonicTheHedgehog-Sms-v0 (top 15 labels) ===
 947  RIGHT
 842  RIGHT+B
 551  RIGHT+A
 177  RIGHT+A+B
 174  NOOP
  91  LEFT
  65  B
  58  LEFT+A
  48  RECOVERY_RIGHT+A+B
  47  A

=== SuperMarioBros-Nes-v0 (top 15 labels) ===
 886  RIGHT
 695  RIGHT+B
 511  RIGHT+A
 272  RIGHT+A+B
 187  NOOP
 126  LEFT
 120  A
  96  RECOVERY_RIGHT+A+B
  80  B
  27  LEFT+A

=== SuperMarioBros3-Nes-v0 (top 15 labels) ===
 838  RIGHT+B
 809  RIGHT
 544  RIGHT+A
 228  RIGHT+A+B
 144  NOOP
 128  B
 126  A


In [5]:
# Inspect one game run step-by-step
GAME = sorted(set(r['game'] for r in records))[0]
EPISODE = 0
N = 40

subset = [
    r for r in records
    if r['game'] == GAME and int(r['episode']) == EPISODE
]
subset = sorted(subset, key=lambda x: int(x['step']))

print(f'Inspecting GAME={GAME}, EPISODE={EPISODE}, rows={len(subset)}')
print('step | action_label | pressed_buttons')
print('-' * 100)
for r in subset[:N]:
    step = int(r['step'])
    label = r['action_label']
    pressed = ','.join(r['pressed_buttons'])
    print(f'{step:4d} | {label:35s} | {pressed}')

Inspecting GAME=AdventureIslandII-Nes-v0, EPISODE=0, rows=1000
step | action_label | pressed_buttons
----------------------------------------------------------------------------------------------------
   0 | RIGHT+A                             | RIGHT,A
   1 | RIGHT+A                             | RIGHT,A
   2 | RIGHT+A                             | RIGHT,A
   3 | RIGHT+A                             | RIGHT,A
   4 | RIGHT+A                             | RIGHT,A
   5 | RIGHT+A                             | RIGHT,A
   6 | RIGHT+A                             | RIGHT,A
   7 | RIGHT+A                             | RIGHT,A
   8 | RIGHT+A                             | RIGHT,A
   9 | RIGHT+A                             | RIGHT,A
  10 | RIGHT+A                             | RIGHT,A
  11 | RIGHT                               | RIGHT
  12 | RIGHT                               | RIGHT
  13 | RIGHT                               | RIGHT
  14 | RIGHT                               | RIGHT
  15 | RIGH

### Test Dataloader

In [8]:


config = TransitionLoaderConfig(
    batch_size=8,
    num_workers=0,
    shuffle_train=True,
    image_size=128,
    normalize="zero_one",
)

loaders, label_map = create_transition_dataloaders(DATASET_DIR, config)

print("Loaded splits:", list(loaders.keys()))
print("Num action labels:", len(label_map))
print("Label sample:", list(sorted(label_map.items()))[:10])

print("\nInspecting one batch from the training loader:\n")

batch = next(iter(loaders["train"]))

print("Batch keys:", list(batch.keys()))
print("\tframe_t shape:", tuple(batch["frame_t"].shape), "frames are likely in (B, C, H, W) format")
print("\tframe_tp1 shape:", tuple(batch["frame_tp1"].shape))
print("\tframe_pair shape:", tuple(batch["frame_pair"].shape))
# print("\taction_id shape:", tuple(batch["action_id"].shape))
print(f"\taction_id = {batch['action_id']} \n\taction_vector shape:", tuple(batch["action_vector"].shape))
print("\tgames sample:", batch["game"][:3])

print("\nBatch action IDs:", batch["action_id"])
print("Batch action labels:", [label for label, idx in label_map.items() if idx in batch["action_id"]])
# print("Batch action tokens (string):", batch["action_tokens"])

print("\nBatch action vectors (one-hot):")
print(batch["action_vector"]) 

print("\nBatch frame pair shape:", batch["frame_pair"].shape)
print("Batch frame pair dtype:", batch["frame_pair"].dtype)


Loaded splits: ['train', 'val', 'test']
Num action labels: 10
Label sample: [('A', 0), ('B', 1), ('LEFT', 2), ('LEFT+A', 3), ('NOOP', 4), ('RECOVERY_RIGHT+A+B', 5), ('RIGHT', 6), ('RIGHT+A', 7), ('RIGHT+A+B', 8), ('RIGHT+B', 9)]

Inspecting one batch from the training loader:

Batch keys: ['frame_t', 'frame_tp1', 'frame_pair', 'action_label', 'action_id', 'action_vector', 'game', 'system', 'episode', 'step']
	frame_t shape: (8, 3, 128, 128) frames are likely in (B, C, H, W) format
	frame_tp1 shape: (8, 3, 128, 128)
	frame_pair shape: (8, 2, 3, 128, 128)
	action_id = tensor([9, 6, 0, 8, 7, 8, 7, 9]) 
	action_vector shape: (8, 9)
	games sample: ['SonicTheHedgehog-Sms-v0', 'NinjaGaiden-Nes-v0', 'SonicTheHedgehog-Sms-v0']

Batch action IDs: tensor([9, 6, 0, 8, 7, 8, 7, 9])
Batch action labels: ['A', 'RIGHT', 'RIGHT+A', 'RIGHT+A+B', 'RIGHT+B']

Batch action vectors (one-hot):
tensor([[1., 0., 0., 0., 0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 0., 0., 0., 1., 0.],
        [0., 0., 0., 0.,